# MavenFlix | Subscription Data Understanding & Profiling

**Project:** Subscription Revenue & Retention Analysis

**Phase:** 2 - Data Understanding & Profiling

**Tools:** Pythons - pandas

**Dataset:** Mavenflix subscription records (fictional business case)

**Status:** In Progress

>**Purpose:** Assess the structure and initial quality of the subscription dataset before defining churn, retention, and subscription-growth metrics.

## Executive Overview

## 1. Business context and analytical scope

**Business question:** Can the source data support reliable analysis of subscription growth, cancelllation trends, and retention cohorts?

**Working unit of observation:** Each row appears to describe a subscription record associated with a customer.

## 2. Load the dataset
**Objective:** Transform the original dataset into a workable pandas DataFrame.

In [1]:
#Importing packages
import pandas as pd

In [2]:
#Importing data
subscriptions_df = pd.read_csv(r"/workspaces/retention-analysis/raw/mavenflix.csv")

## 3. Initial structure and schema
**Objective:** Preview the source records, confirm dimensions and column names, and inspects inferred types. Identify which fields need data validation before the analysis.

In [3]:
# Data Overview
display(subscriptions_df.head(10))
print("\n", subscriptions_df.shape)
print("\n", subscriptions_df.columns)
subscriptions_df.info()

,customer_id,created_date,canceled_date,subscription_cost,subscription_interval,was_subscription_paid
0,154536156,2022-09-01,NaN,39,month,Yes
1,149713408,2022-09-01,2022-09-02,39,month,No
2,153756284,2022-09-01,2022-09-02,39,month,No
3,121253113,2022-09-01,2022-09-23,39,month,Yes
4,154467210,2022-09-01,2023-06-29,39,month,Yes
5,154510811,2022-09-01,2023-05-08,39,month,Yes
6,154494346,2022-09-01,2022-11-26,39,month,Yes
7,148671505,2022-09-01,2022-11-23,39,month,Yes
8,154568379,2022-09-01,2022-10-25,39,month,Yes
9,154428434,2022-09-01,2022-10-31,39,month,Yes



 (3069, 6)

 Index(['customer_id', 'created_date', 'canceled_date', 'subscription_cost',
       'subscription_interval', 'was_subscription_paid'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 3069 entries, 0 to 3068
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   customer_id            3069 non-null   int64
 1   created_date           3069 non-null   str  
 2   canceled_date          2004 non-null   str  
 3   subscription_cost      3069 non-null   int64
 4   subscription_interval  3069 non-null   str  
 5   was_subscription_paid  3069 non-null   str  
dtypes: int64(2), str(4)
memory usage: 144.0 KB


### Key Findings
The data contains **3,069 rows and 6 columns**.

| Field | Data Type | Interpretation |
| --- | --- | --- |
| `customer_id` | `int64` | Customer identifier |
| `created_date` | `str` | Subscription start date; need datetime conversion |
| `canceled_date` | `str` | Cancellation date; need datetime conversion |
| `subscription_cost` | `int64` | Subscription price |
| `subscription_interval` | `str` | Billing interval category |
| `was_subscription_paid` | `str` | Recorded payment-status category |

**Note:** One row represents a subscription record linked to a customer. Repeated customer IDs must be examined before interpreting subscription counts as customer counts.

## 4. Missing-value profiling
**Objective:** Statiscal overview of missing data

In [4]:
#Missing Data
missing_count = subscriptions_df.isna().sum()
missing_pct = (subscriptions_df.isna().sum()/subscriptions_df.shape[0]) * 100
missing_df = pd.DataFrame(
    {"Missing Count": missing_count,
     "Missing %": missing_pct}
)
display(missing_df)

,Missing Count,Missing %
customer_id,0,0.000000
created_date,0,0.000000
canceled_date,1065,34.701857
subscription_cost,0,0.000000
subscription_interval,0,0.000000
was_subscription_paid,0,0.000000


### Key Findings
- `canceled_date` is the only field with missing values: **1,065 records (34.70%)**.
- All other fields have 0 missing values.
- Missing cancellation date may be an expected state for subscription without record cancellation.

## 5. Exact duplicates and repeated customer IDs
**Objective:**
- Do any full rows match exactly?
- Are individual customers represented by multiple subscription records?

### 5.1. Exact duplicate rows

In [5]:
#Duplication
print("Duplicated records:", subscriptions_df.duplicated().sum())

Duplicated records: 0


## 5. Duplicate and repeated-customer checks
**Objective:** Check for identical records and repeated customer IDs before interpreting subscription counts as customer counts.

In [6]:
#Repeated customer
print("Unique customers:",subscriptions_df['customer_id'].nunique())

repeated_count = subscriptions_df['customer_id'].value_counts()
print("Customers with multiple records:",repeated_count[repeated_count > 1].shape[0])

print("Maximum records per customer:",repeated_count.max())

Unique customers: 2877
Customers with multiple records: 185
Maximum records per customer: 3


### Key findings
- **0** exact duplicate rows across six source fields.
- **2,877** distinct customer IDs; **185** appear more than once, with a maximum of **3** records per customers.

## 6. Categorical value profiling
**Objective:** Examine distinct billing intervals and recorded payment-status categories, including any missing groups.

In [7]:
#Categorical Data Profiling
print(subscriptions_df.groupby('subscription_interval',dropna=False).size())
print(subscriptions_df.groupby('was_subscription_paid',dropna=False).size())

subscription_interval
month    3069
dtype: int64
was_subscription_paid
No      133
Yes    2936
dtype: int64


### Key Findings
| Field | Category | Saved record count |
| --- | --- | --- |
| `subscription_interval` | `month` | 3,069 |
| `was_subscription_paid` | `Yes` | 2,936 |
| `was_subscription_paid` | `No` | 133 |

All records are labled with the `month` interval. Payment status contains two labels `Yes` and `No`

## 7. Date Validity Analysis
**Objective:** Ensure subscription dates are reliable and in the correct format and chronological consistency.

In [8]:
#Checking data types of the two date columns
print(subscriptions_df['created_date'].dtype)
print(subscriptions_df['canceled_date'].dtype)

str
str


In [9]:
#Create new columns with datetime data type
subscriptions_df['created_date_dt'] = pd.to_datetime(subscriptions_df['created_date'], format='%Y-%m-%d') 
subscriptions_df['canceled_date_dt'] = pd.to_datetime(subscriptions_df['canceled_date'], format='%Y-%m-%d', errors='coerce')

In [10]:
print('Earliest subscription date:',subscriptions_df['created_date_dt'].min())
print('Latest subscription date:',subscriptions_df['created_date_dt'].max())
print('Earliest cancellation date:',subscriptions_df['canceled_date_dt'].min())
print('Latest cancellation date:',subscriptions_df['canceled_date_dt'].max())
print('Invalid creation dates:',subscriptions_df['created_date_dt'].isna().sum())
invalid_canceled = (
    subscriptions_df['canceled_date'].notna() & subscriptions_df['canceled_date_dt'].isna()
)
print('Invalid cancellation dates:',invalid_canceled.sum())

Earliest subscription date: 2022-09-01 00:00:00
Latest subscription date: 2023-09-08 00:00:00
Earliest cancellation date: 2022-09-02 00:00:00
Latest cancellation date: 2023-09-08 00:00:00
Invalid creation dates: 0
Invalid cancellation dates: 0


In [11]:
#Chronological validity
invalid_date_mask = (
    subscriptions_df['canceled_date_dt']
    < subscriptions_df['created_date_dt']
)

invalid_dates_df = subscriptions_df[invalid_date_mask]

print("Invalid date sequences:", invalid_dates_df.shape[0])

display(invalid_dates_df)

Invalid date sequences: 0


,customer_id,created_date,canceled_date,subscription_cost,subscription_interval,was_subscription_paid,created_date_dt,canceled_date_dt


### Key Findings
- **Date range:** Subscription creation dates range from September 1st, 2022, to September 8th, 2023, while cancellation dates range from September 2nd, 2022, to September 8th, 2023.
- **Data validity:** No datetime conversion failures were detected in either column. Missing cancellation dates were excluded from the invalid date count.
- **Chronological consistency:** No record of cancellation dates earlier than creation dates was detected.
- **Missing cancellation dates:** Subscriptions without recorded cancellations are not classified as data-quality errors.


## 8. Numerical Value Validation
**Objectives:** Assess the distribution and validity of subscription costs before performing subscription-value analysis

In [12]:
#Descriptive statistics of the numeric columns
print("Minimum subscription cost:", subscriptions_df['subscription_cost'].min())
print("Maximum subscription cost:", subscriptions_df['subscription_cost'].max())
print("Mean subscription cost:", subscriptions_df['subscription_cost'].mean())
print("Median subscription cost:", subscriptions_df['subscription_cost'].median())
print("Subscription cost equal or below zero:", (subscriptions_df['subscription_cost'] <= 0).sum())

Minimum subscription cost: 39
Maximum subscription cost: 39
Mean subscription cost: 39.0
Median subscription cost: 39.0
Subscription cost equal or below zero: 0


### Key Findings
- All 3,069 records have the uniform subscription cost of 39.
- No zero or negative subscription cost detected.
- No variation or statistical outliers were detected.

## 9. Data Profiling Summary

### 9.1. Dataset Overview
The MavenFlix dataset contains 3,069 subscription records across six attributes. Initial profiling identified no exact duplicate rows. All subscriptions follow a monthly billing inverval with a uniform subscription price of 39. The dataset contains 2,877 unique customer IDs, indicating that some customers have multiple subscription records.

### 9.2. Key Data Quality Findings
Approximately 35% of records have missing cancellation dates, which may represent subscriptions without recorded cancellations rather than data-quality errors.

Repeated customer IDs were identified, including records consistent with re-subscription activity. These records require careful consideration when distinguishing subscription-level metrics from customer-level metrics.

Date validation detected no conversion failures, while subscription costs were consistently positive. However, these checks alone do not establish that all subscription histories are logically consistent.

### 9.3. Readiness Assessment
The dataset appears suitable for further preperation and exploratory subscription analysis, subsject to clarification of its business rules and remaining data-quality checks.

Before calculating churn and cohort retention, an observation cutoff must be established, and consistent definitions for subscription activity, cancellation, and retention must be documented.

Additionally, payment-status indicators should not be interpreted as complete record of collected revenue.